In [1]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt # 💡 THE FIX
import seaborn as sns

# --- 1. CONFIGURATION & SETUP ---
# Dynamic resolution mapping based on Nandana et al. (2026) Table 3
def get_resolution(ex_num):
    if ex_num == 2:
        return 810.0, 1800.0 # Ex 2 (Extending Elbow) is 810x1800
    else:
        return 810.0, 1440.0 # Ex 1, 3, 4 are 810x1440

optimal_params = {
    1: {"span": 3, "threshold": 0.08, "joints": [11, 12, 13, 14, 15, 16]},
    2: {"span": 3, "threshold": 0.08, "joints": [11, 12, 13, 14, 15, 16]},
    3: {"span": 3, "threshold": 0.08, "joints": [11, 12, 13, 14, 15, 16]},
    4: {"span": 3, "threshold": 0.08, "joints": [11, 12, 13, 14, 15, 16]} 
}

final_results = {"YOLO_Raw": [], "YOLO_Filter": [], "MP_Raw": [], "MP_Filter": []}
global_mp_predictions = {1: {}, 2: {}, 3: {}, 4: {}}

# Update this path to where you extracted the dataset on your Mac
base_data_path = "/Users/sofiasaifulrizal/Desktop/An_upper_limb_stroke_rehabilitation_exercise_video/Exercise"

print("🚀 RUNNING NAIR ET AL. (2026) MASTER BATCH EVALUATION...\n")

# --- 2. CORE EVALUATION LOOP ---
for ex_num in [1, 2, 3, 4]: 
    
    # Fetch the exact camera resolution for this specific exercise
    FRAME_WIDTH, FRAME_HEIGHT = get_resolution(ex_num)
    
    opt_span = optimal_params[ex_num]["span"]
    opt_thresh = optimal_params[ex_num]["threshold"]
    joints_to_dtw = optimal_params[ex_num]["joints"]

    for fw_name in ["MediaPipe", "YOLOv8"]: 
        folder_path = f"{base_data_path}/{'MediaPipe_CSV' if fw_name == 'MediaPipe' else 'YOLO_CSV'}/Ex{ex_num}"
        
        for run_level in [1, 4]: 
            correct_preds = 0
            total_vids = 0
            
            all_files = glob.glob(os.path.join(folder_path, "*.csv"))
            if not all_files: continue
            
            # 💡 EXPERT SELECTION: Find the first 'Complete' video (Completion Status == '01')
            expert_file = next((f for f in all_files if os.path.basename(f).split('_')[2] == '01'), None)
            if not expert_file: continue
            
            raw_expert_df = pd.read_csv(expert_file)
            
            if fw_name == "YOLOv8": 
                raw_expert_df.rename(columns=yolo_to_mp_map, inplace=True)
                for c in yolo_to_mp_map.values():
                    if c in raw_expert_df.columns:
                        col_data = raw_expert_df[c].astype(float)
                        # Absolute Resolution Scaling mapped to Mendeley cameras
                        if '_x' in c: raw_expert_df[c] = col_data / FRAME_WIDTH
                        elif '_y' in c: raw_expert_df[c] = col_data / FRAME_HEIGHT

            expert_df = apply_ablation_filter(raw_expert_df, run_level, ex_num, opt_span)
            patient_files = [f for f in all_files if f != expert_file]
            
            for p_file in patient_files: 
                filename = os.path.basename(p_file)
                # 💡 GROUND TRUTH EXTRACTION: Parse directly from VV_EE_CC_NN.csv
                parts = filename.replace('.csv', '').split('_')
                if len(parts) < 3: continue
                true_status = 'complete' if parts[2] == '01' else 'incomplete'
                
                try:
                    raw_pat_df = pd.read_csv(p_file)
                    
                    if fw_name == "YOLOv8": 
                        raw_pat_df.rename(columns=yolo_to_mp_map, inplace=True)
                        for c in yolo_to_mp_map.values():
                            if c in raw_pat_df.columns:
                                col_data = raw_pat_df[c].astype(float)
                                # Absolute Resolution Scaling mapped to Mendeley cameras
                                if '_x' in c: raw_pat_df[c] = col_data / FRAME_WIDTH
                                elif '_y' in c: raw_pat_df[c] = col_data / FRAME_HEIGHT

                    rep_df = apply_ablation_filter(raw_pat_df, run_level, ex_num, opt_span)
                    if rep_df.empty: continue

                    # Evaluate DTW
                    raw_score = evaluate_video_with_dtw(rep_df, expert_df, active_joints=joints_to_dtw, threshold=opt_thresh)
                    result_code = 'complete' if raw_score == 1 else 'incomplete'
                    
                    # Tally accuracy
                    total_vids += 1
                    if result_code == true_status:
                        correct_preds += 1
                    
                    if fw_name == "MediaPipe" and run_level == 4:
                        global_mp_predictions[ex_num][filename] = result_code
                        
                except Exception as e: 
                    pass
            
            # Calculate final accuracy percentage for this run
            acc = round((correct_preds / total_vids) * 100, 1) if total_vids > 0 else 0.0
            
            if fw_name == "MediaPipe" and run_level == 1: final_results["MP_Raw"].append(acc)
            elif fw_name == "MediaPipe" and run_level == 4: final_results["MP_Filter"].append(acc)
            elif fw_name == "YOLOv8" and run_level == 1: final_results["YOLO_Raw"].append(acc)
            elif fw_name == "YOLOv8" and run_level == 4: final_results["YOLO_Filter"].append(acc)

# --- 3. VISUALIZATION (HORIZONTAL BAR CHART) ---
data = []
exercises = [
    'Exercise 1\n(Lifting an Object)', 
    'Exercise 2\n(Extending the Elbow)', 
    'Exercise 3\n(Lifting the Wrist)', 
    'Exercise 4\n(Opening the Hand)'
]

# 💡 Applying the 40.0% Test Accuracy Baseline from Nair et al. 2026 (Table 7)
sota_scores = [40.0, 40.0, 40.0, 40.0] 

for i, ex_name in enumerate(exercises): 
    data.append({'Exercise': ex_name, 'Pipeline': 'YOLOv8 (Raw)', 'Accuracy': final_results["YOLO_Raw"][i]})
    data.append({'Exercise': ex_name, 'Pipeline': 'YOLOv8 + Strict Filter', 'Accuracy': final_results["YOLO_Filter"][i]})
    data.append({'Exercise': ex_name, 'Pipeline': 'MediaPipe (Raw)', 'Accuracy': final_results["MP_Raw"][i]})
    data.append({'Exercise': ex_name, 'Pipeline': 'MediaPipe + Strict Filter', 'Accuracy': final_results["MP_Filter"][i]})
    data.append({'Exercise': ex_name, 'Pipeline': 'Nair et al. 2026 (3D CNN Baseline)', 'Accuracy': sota_scores[i]})

df_plot = pd.DataFrame(data)
fig, ax = plt.subplots(figsize=(12, 10)) 
colors = ['#f2dede', '#d9534f', '#dff0d8', '#5cb85c', '#5bc0de'] 

bars = sns.barplot(data=df_plot, y='Exercise', x='Accuracy', hue='Pipeline', palette=colors, edgecolor='black', linewidth=1.2, ax=ax)

ax.set_title('Vision-based Movement Analysis: Proposed 2D Frameworks vs. 3D CNN Baseline', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Classification Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('', fontsize=12) 
ax.set_xlim(0, 115) 
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=True)

for bar in bars.patches:
    width = bar.get_width()
    if width > 0: 
        ax.annotate(f'{width:.1f}%', xy=(width, bar.get_y() + bar.get_height() / 2), xytext=(5, 0), textcoords="offset points", ha='left', va='center', fontsize=11, fontweight='bold')

sns.despine(left=True, bottom=False) 
plt.show()

🚀 RUNNING NAIR ET AL. (2026) MASTER BATCH EVALUATION...



IndexError: list index out of range